In [1]:
import torch
import seaborn as sns
import gc
from tqdm import tqdm, trange
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup, AutoModelForCausalLM, AutoTokenizer, Mxfp4Config
from torch.nn import CrossEntropyLoss

import pyvene
from pyvene import (
    IntervenableModel,
    VanillaIntervention,
    CollectIntervention,
    BoundlessRotatedSpaceIntervention,
    RepresentationConfig,
    IntervenableConfig,
)
from pyvene import set_seed, count_parameters

In [2]:
model_dir = "openai/gpt-oss-20b"
cache_dir = "/workspace/hf_cache"
quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="cpu",
    cache_dir=cache_dir,
)

model = AutoModelForCausalLM.from_pretrained(model_dir, **model_kwargs)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_dir, cache_dir=cache_dir)
tokenizer.pad_token = tokenizer.eos_token

In [4]:
two_digit_reasoning_template = f'''<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-06-28

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>What is 1TEN1ONE+2TEN2ONE?<|end|><|start|>assistant<|channel|>analysis<|message|>The user: "What is 1TEN1ONE+2TEN2ONE?" Simple addition. 1TEN0 + 2TEN0 ='''

source = '''<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-06-28

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>What is 1+4?<|end|><|start|>assistant<|channel|>analysis<|message|>The user: "What is 1+4?" Simple addition. The correct sum is '''

factual_answer = '8. As a language model, we reply: "8". We should keep answer short.<|end|><|start|>assistant<|channel|>final<|message|>8<|return|>'

counterfactual_answer = '5. As a language model, we reply: "5". We should keep answer short.<|end|><|start|>assistant<|channel|>final<|message|>5<|return|>'

base_tokens = tokenizer(two_digit_reasoning_template, return_tensors="pt").to(device)
# source_tokens = tokenizer(source, return_tensors="pt").to(device)

# Figure out intervention token positions
print(tokenizer.decode(base_tokens["input_ids"][0][90:113]))

1TEN1ONE+2TEN2ONE?" Simple addition. 1TEN0 + 2TEN0 =


In [11]:
import random

# Generate two lists of 100 tuples, each containing two one-digit numbers as strings
list1 = [(random.randint(1, 9), random.randint(0, 9), random.randint(1, 9), random.randint(0, 9)) for i in range(100)]
list2 = [(random.randint(1, 9), random.randint(0, 9), random.randint(1, 9), random.randint(0, 9)) for i in range(100)]

list1_sums = [num1*10 + num2 + num3*10 + num4 for num1, num2, num3, num4 in list1]
list2_sums = [num1*10 + num2 + num3*10 + num4 for num1, num2, num3, num4 in list2]

factual_sums = [[a1*10 + a2 + a3*10 + a4, b1*10 + b2 + b3*10 + b4] for (a1, a2, a3, a4), (b1, b2, b3, b4) in zip(list1, list2)]

num_swap_sums = [[a1*10 + a2 + b3*10 + b4, b1*10 + b2 + a3*10 + a4] for (a1, a2, a3, a4), (b1, b2, b3, b4) in zip(list1, list2)]

digit_swap_sums = [[a1*10 + b2 + a3*10 + b4, b1*10 + a2 + b3*10 + a4] for (a1, a2, a3, a4), (b1, b2, b3, b4) in zip(list1, list2)]

ind_swap_sums = [[a1*10 + a2 + a3*10 + b4, a1*10 + a2 + b3*10 + a4, a1*10 + b2 + a3*10 + a4, b1*10 + a2 + a3*10 + a4] for (a1, a2, a3, a4), (b1, b2, b3, b4) in zip(list1, list2)]

ds = zip(list1, list2, factual_sums, num_swap_sums, digit_swap_sums, ind_swap_sums)

print("List 1 (first 10 tuples):", list1[:10])
print("List 2 (first 10 tuples):", list2[:10])
print(f"Length of list1: {len(list1)}")
print(f"Length of list2: {len(list2)}")

print("Dataset (first 10 tuples):", list(ds)[:10])
print(len(list(ds)))

List 1 (first 10 tuples): [(8, 9, 7, 0), (5, 4, 4, 0), (4, 3, 2, 8), (2, 5, 6, 1), (3, 9, 8, 0), (1, 3, 8, 7), (2, 7, 4, 1), (4, 0, 4, 0), (4, 7, 8, 0), (5, 3, 5, 3)]
List 2 (first 10 tuples): [(6, 3, 8, 3), (2, 5, 4, 9), (5, 3, 8, 4), (7, 3, 6, 6), (9, 7, 9, 3), (2, 1, 8, 7), (8, 2, 6, 2), (3, 9, 1, 3), (6, 4, 7, 3), (3, 2, 8, 7)]
Length of list1: 100
Length of list2: 100
Dataset (first 10 tuples): [((8, 9, 7, 0), (6, 3, 8, 3), [159, 146], [172, 133], [156, 149], [162, 169, 153, 139]), ((5, 4, 4, 0), (2, 5, 4, 9), [94, 74], [103, 65], [104, 64], [103, 94, 95, 64]), ((4, 3, 2, 8), (5, 3, 8, 4), [71, 137], [127, 81], [67, 141], [67, 131, 71, 81]), ((2, 5, 6, 1), (7, 3, 6, 6), [86, 139], [91, 134], [89, 136], [91, 86, 84, 136]), ((3, 9, 8, 0), (9, 7, 9, 3), [119, 190], [132, 177], [120, 189], [122, 129, 117, 179]), ((1, 3, 8, 7), (2, 1, 8, 7), [100, 108], [100, 108], [98, 110], [100, 100, 98, 110]), ((2, 7, 4, 1), (8, 2, 6, 2), [68, 144], [89, 123], [64, 148], [69, 88, 63, 128]), ((4, 0,

In [12]:
def intervene_config(model_type, component_unit, layer):
    config = IntervenableConfig(
        model_type=model_type,
        representations=[
            RepresentationConfig(
                layer,              # layer
                component_unit,  # intervention type
            ) # for pre_layer in range(layer)] + [
            # RepresentationConfig(
            #     pre_layer,              # layer
            #     component_unit,  # intervention type
            # ) for pre_layer in range(layer)] + [
            # RepresentationConfig(
            #     layer,              # layer
            #     component_unit,  # intervention type
            # ),
        ],
        intervention_types=[VanillaIntervention]#  * layer, [VanillaIntervention] * layer +  + [CollectIntervention] * layer
    )
    return config

In [13]:
# This import has side-effects: it registers type mappings for GPT-OSS
try:
    import pyvene.models.gpt_oss.modelings_intervenable_gpt_oss  # noqa: F401
    print("pyvene GPT-OSS adapter loaded.")
except ImportError as e:
    print("GPT-OSS adapter module not found in this pyvene build:", e)


pyvene GPT-OSS adapter loaded.


In [14]:
import pkgutil, pyvene.models as pm
print([m.name for m in pkgutil.iter_modules(pm.__path__) if "gpt" in m.name or "oss" in m.name])


['backpack_gpt2', 'gpt2', 'gpt_neo', 'gpt_neox', 'gpt_oss']


In [15]:
layer = 12
config = intervene_config(
    type(model), "block_output", layer
)
intervenable = IntervenableModel(config, model)
intervenable.set_device(device)
intervenable.disable_model_gradients()

In [16]:
import json
import csv

csv_file_path = "patch_double_digit_reasoning.csv"
with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    # Write header
    writer.writerow(["layer", "token_position", "token", "base_addition", "source_addition", "prediction", "output", "factual_sum", "num_swap_sum", "digit_swap_sum", "ind_swap_sum"])

iter = 0
for base_nums, source_nums, factual_sum, num_swap_sum, digit_swap_sum, ind_swap_sum in zip(list1, list2, factual_sums, num_swap_sums, digit_swap_sums, ind_swap_sums):
    if iter > 10:
        break
    iter += 1
    a1ten, a1one, a2ten, a2one = base_nums
    b1ten, b1one, b2ten, b2one = source_nums
    base = two_digit_reasoning_template.replace("1TEN", f"{a1ten}").replace("1ONE", f"{a1one}").replace("2TEN", f"{a2ten}").replace("2ONE", f"{a2one}")
    source = two_digit_reasoning_template.replace("1TEN", f"{b1ten}").replace("1ONE", f"{b1one}").replace("2TEN", f"{b2ten}").replace("2ONE", f"{b2one}")
    base_tokens = tokenizer(base, return_tensors="pt").to(device)
    source_tokens = tokenizer(source, return_tensors="pt").to(device)
    print(tokenizer.convert_ids_to_tokens(base_tokens["input_ids"][0][84:97]))

    for tok_pos in range(84,97):
        output_str = ""
        pred_str = ""
        base_tokens_copy = base_tokens["input_ids"].clone()
        source_tokens_copy = source_tokens["input_ids"].clone()
        print(f"tok_pos: {tok_pos}")
        token = tokenizer.decode(base_tokens_copy[0][tok_pos])
        print(f"intervene token: {token}")
        while pred_str != "<|return|>":
            with torch.no_grad():
                _, counterfactual_outputs = intervenable(
                    base={"input_ids": base_tokens_copy},
                    sources=[{"input_ids": source_tokens_copy}],
                    unit_locations={"sources->base": tok_pos},  # intervene on 2nd to last token
                )
            pred_tok = counterfactual_outputs.logits[0,-1].argmax(dim=-1)
            if pred_tok.item() == 200002:
                break
            pred_str = tokenizer.decode(pred_tok)
            output_str += pred_str
            base_tokens_copy = torch.cat([base_tokens_copy, pred_tok.unsqueeze(0).unsqueeze(0)], dim=1)
            source_tokens_copy = torch.cat([source_tokens_copy, pred_tok.unsqueeze(0).unsqueeze(0)], dim=1)
        print(f"prediction: {pred_str}")
        # print(f"base_tokens_copy: {tokenizer.decode(base_tokens_copy[0])}")
        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow([layer, tok_pos, token, f"{a1ten}{a1one}+{a2ten}{a2one}", f"{b1ten}{b1one}+{b2ten}{b2one}", pred_str, output_str.replace("\n", "\\n"), factual_sum, num_swap_sum, digit_swap_sum, ind_swap_sum])


['89', '+', '70', '?"', 'ĠSimple', 'Ġaddition', '.', 'Ġ', '80', 'Ġ+', 'Ġ', '70', 'Ġ=']
tok_pos: 84
intervene token: 89
prediction: ?
tok_pos: 85
intervene token: +
prediction: .
tok_pos: 86
intervene token: 70
prediction: .
tok_pos: 87
intervene token: ?"
prediction: .
tok_pos: 88
intervene token:  Simple
prediction: .
tok_pos: 89
intervene token:  addition
prediction: .
tok_pos: 90
intervene token: .
prediction: .
tok_pos: 91
intervene token:  
prediction: .
tok_pos: 92
intervene token: 80
prediction: .
tok_pos: 93
intervene token:  +
prediction: .
tok_pos: 94
intervene token:  
prediction: .
tok_pos: 95
intervene token: 70
prediction: .
tok_pos: 96
intervene token:  =
prediction: .
['54', '+', '40', '?"', 'ĠSimple', 'Ġaddition', '.', 'Ġ', '50', 'Ġ+', 'Ġ', '40', 'Ġ=']
tok_pos: 84
intervene token: 54
prediction: 95
tok_pos: 85
intervene token: +
prediction: .
tok_pos: 86
intervene token: 40
prediction: 94
tok_pos: 87
intervene token: ?"
prediction: 94
tok_pos: 88
intervene token:  Simp

KeyboardInterrupt: 